# 4 — Train Model / 4 Train Model

**Chapter 26 — File 4 of 7 / 第26章 — 第4个文件（共7个）**

---

## Summary / 总结

This script demonstrates **load doc into memory**.

本脚本演示 **load doc into memory**。

---
## Step 1 — Step 1

In [ ]:
from numpy import array
from pickle import load
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.utils import plot_model
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding
from keras.layers import Dropout
from keras.layers.merge import add
from keras.callbacks import ModelCheckpoint

---
## Step 2 — load doc into memory

In [ ]:
def load_doc(filename):

---
## Step 3 — open the file as read only

In [ ]:
file = open(filename, 'r')

---
## Step 4 — read all text

In [ ]:
text = file.read()

---
## Step 5 — close the file

In [ ]:
file.close()
	return text

---
## Step 6 — load a pre-defined list of photo identifiers

In [ ]:
def load_set(filename):
	doc = load_doc(filename)
	dataset = list()

---
## Step 7 — process line by line

In [ ]:
for line in doc.split('\n'):

---
## Step 8 — skip empty lines

In [ ]:
if len(line) < 1:
			continue

---
## Step 9 — get the image identifier

In [ ]:
identifier = line.split('.')[0]
		dataset.append(identifier)
	return set(dataset)

---
## Step 10 — load clean descriptions into memory

In [ ]:
def load_clean_descriptions(filename, dataset):

---
## Step 11 — load document

In [ ]:
doc = load_doc(filename)
	descriptions = dict()
	for line in doc.split('\n'):

---
## Step 12 — split line by white space

In [ ]:
tokens = line.split()

---
## Step 13 — split id from description

In [ ]:
image_id, image_desc = tokens[0], tokens[1:]

---
## Step 14 — skip images not in the set

In [ ]:
if image_id in dataset:

---
## Step 15 — create list

In [ ]:
if image_id not in descriptions:
				descriptions[image_id] = list()

---
## Step 16 — wrap description in tokens

In [ ]:
desc = 'startseq ' + ' '.join(image_desc) + ' endseq'

---
## Step 17 — store

In [ ]:
descriptions[image_id].append(desc)
	return descriptions

---
## Step 18 — load photo features

In [ ]:
def load_photo_features(filename, dataset):

---
## Step 19 — load all features

In [ ]:
all_features = load(open(filename, 'rb'))

---
## Step 20 — filter features

In [ ]:
features = {k: all_features[k] for k in dataset}
	return features

---
## Step 21 — covert a dictionary of clean descriptions to a list of descriptions

In [ ]:
def to_lines(descriptions):
	all_desc = list()
	for key in descriptions.keys():
		[all_desc.append(d) for d in descriptions[key]]
	return all_desc

---
## Step 22 — fit a tokenizer given caption descriptions

In [ ]:
def create_tokenizer(descriptions):
	lines = to_lines(descriptions)
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

---
## Step 23 — calculate the length of the description with the most words

In [ ]:
def max_length(descriptions):
	lines = to_lines(descriptions)
	return max(len(d.split()) for d in lines)

---
## Step 24 — create sequences of images, input sequences and output words for an image

In [ ]:
def create_sequences(tokenizer, max_length, descriptions, photos, vocab_size):
	X1, X2, y = list(), list(), list()

---
## Step 25 — walk through each image identifier

In [ ]:
for key, desc_list in descriptions.items():

---
## Step 26 — walk through each description for the image

In [ ]:
for desc in desc_list:

---
## Step 27 — encode the sequence

In [ ]:
seq = tokenizer.texts_to_sequences([desc])[0]

---
## Step 28 — split one sequence into multiple X,y pairs

In [ ]:
for i in range(1, len(seq)):

---
## Step 29 — split into input and output pair

In [ ]:
in_seq, out_seq = seq[:i], seq[i]

---
## Step 30 — pad input sequence

In [ ]:
in_seq = pad_sequences([in_seq], maxlen=max_length)[0]

---
## Step 31 — encode output sequence

In [ ]:
out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

---
## Step 32 — store

In [ ]:
X1.append(photos[key][0])
				X2.append(in_seq)
				y.append(out_seq)
	return array(X1), array(X2), array(y)

---
## Step 33 — define the captioning model

In [ ]:
def define_model(vocab_size, max_length):

---
## Step 34 — feature extractor model

In [ ]:
inputs1 = Input(shape=(4096,))
	fe1 = Dropout(0.5)(inputs1)
	fe2 = Dense(256, activation='relu')(fe1)

---
## Step 35 — sequence model

In [ ]:
inputs2 = Input(shape=(max_length,))
	se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
	se2 = Dropout(0.5)(se1)
	se3 = LSTM(256)(se2)

---
## Step 36 — decoder model

In [ ]:
decoder1 = add([fe2, se3])
	decoder2 = Dense(256, activation='relu')(decoder1)
	outputs = Dense(vocab_size, activation='softmax')(decoder2)

---
## Step 37 — tie it together [image, seq] [word]

In [ ]:
model = Model(inputs=[inputs1, inputs2], outputs=outputs)

---
## Step 38 — compile model

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam')

---
## Step 39 — summarize model

In [ ]:
model.summary()
	plot_model(model, to_file='model.png', show_shapes=True)
	return model

---
## Step 40 — load training dataset (6K)

In [ ]:
filename = 'Flickr8k_text/Flickr_8k.trainImages.txt'
train = load_set(filename)
print('Dataset: %d' % len(train))

---
## Step 41 — descriptions

In [ ]:
train_descriptions = load_clean_descriptions('descriptions.txt', train)
print('Descriptions: train=%d' % len(train_descriptions))

---
## Step 42 — photo features

In [ ]:
train_features = load_photo_features('features.pkl', train)
print('Photos: train=%d' % len(train_features))

---
## Step 43 — prepare tokenizer

In [ ]:
tokenizer = create_tokenizer(train_descriptions)
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary Size: %d' % vocab_size)

---
## Step 44 — determine the maximum sequence length

In [ ]:
max_length = max_length(train_descriptions)
print('Description Length: %d' % max_length)

---
## Step 45 — prepare sequences

In [ ]:
X1train, X2train, ytrain = create_sequences(tokenizer, max_length, train_descriptions, train_features, vocab_size)

---
## Step 46 — load test set

In [ ]:
filename = 'Flickr8k_text/Flickr_8k.devImages.txt'
test = load_set(filename)
print('Dataset: %d' % len(test))

---
## Step 47 — descriptions

In [ ]:
test_descriptions = load_clean_descriptions('descriptions.txt', test)
print('Descriptions: test=%d' % len(test_descriptions))

---
## Step 48 — photo features

In [ ]:
test_features = load_photo_features('features.pkl', test)
print('Photos: test=%d' % len(test_features))

---
## Step 49 — prepare sequences

In [ ]:
X1test, X2test, ytest = create_sequences(tokenizer, max_length, test_descriptions, test_features, vocab_size)

---
## Step 50 — define the model

In [ ]:
model = define_model(vocab_size, max_length)

---
## Step 51 — define checkpoint callback

In [ ]:
checkpoint = ModelCheckpoint('model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='min')

---
## Step 52 — fit model

In [ ]:
model.fit([X1train, X2train], ytrain, epochs=20, verbose=2, callbacks=[checkpoint], validation_data=([X1test, X2test], ytest))

---
## Learning Notes / 学习笔记

- **概念**: load doc into memory 是机器学习中的常用技术。  
  *load doc into memory is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Model / 4 Train Model
# Complete Code / 完整代码
# ===============================

from numpy import array
from pickle import load
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.utils import plot_model
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding
from keras.layers import Dropout
from keras.layers.merge import add
from keras.callbacks import ModelCheckpoint

# load doc into memory
def load_doc(filename):
	# open the file as read only
	file = open(filename, 'r')
	# read all text
	text = file.read()
	# close the file
	file.close()
	return text

# load a pre-defined list of photo identifiers
def load_set(filename):
	doc = load_doc(filename)
	dataset = list()
	# process line by line
	for line in doc.split('\n'):
		# skip empty lines
		if len(line) < 1:
			continue
		# get the image identifier
		identifier = line.split('.')[0]
		dataset.append(identifier)
	return set(dataset)

# load clean descriptions into memory
def load_clean_descriptions(filename, dataset):
	# load document
	doc = load_doc(filename)
	descriptions = dict()
	for line in doc.split('\n'):
		# split line by white space
		tokens = line.split()
		# split id from description
		image_id, image_desc = tokens[0], tokens[1:]
		# skip images not in the set
		if image_id in dataset:
			# create list
			if image_id not in descriptions:
				descriptions[image_id] = list()
			# wrap description in tokens
			desc = 'startseq ' + ' '.join(image_desc) + ' endseq'
			# store
			descriptions[image_id].append(desc)
	return descriptions

# load photo features
def load_photo_features(filename, dataset):
	# load all features
	all_features = load(open(filename, 'rb'))
	# filter features
	features = {k: all_features[k] for k in dataset}
	return features

# covert a dictionary of clean descriptions to a list of descriptions
def to_lines(descriptions):
	all_desc = list()
	for key in descriptions.keys():
		[all_desc.append(d) for d in descriptions[key]]
	return all_desc

# fit a tokenizer given caption descriptions
def create_tokenizer(descriptions):
	lines = to_lines(descriptions)
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

# calculate the length of the description with the most words
def max_length(descriptions):
	lines = to_lines(descriptions)
	return max(len(d.split()) for d in lines)

# create sequences of images, input sequences and output words for an image
def create_sequences(tokenizer, max_length, descriptions, photos, vocab_size):
	X1, X2, y = list(), list(), list()
	# walk through each image identifier
	for key, desc_list in descriptions.items():
		# walk through each description for the image
		for desc in desc_list:
			# encode the sequence
			seq = tokenizer.texts_to_sequences([desc])[0]
			# split one sequence into multiple X,y pairs
			for i in range(1, len(seq)):
				# split into input and output pair
				in_seq, out_seq = seq[:i], seq[i]
				# pad input sequence
				in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
				# encode output sequence
				out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
				# store
				X1.append(photos[key][0])
				X2.append(in_seq)
				y.append(out_seq)
	return array(X1), array(X2), array(y)

# define the captioning model
def define_model(vocab_size, max_length):
	# feature extractor model
	inputs1 = Input(shape=(4096,))
	fe1 = Dropout(0.5)(inputs1)
	fe2 = Dense(256, activation='relu')(fe1)
	# sequence model
	inputs2 = Input(shape=(max_length,))
	se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
	se2 = Dropout(0.5)(se1)
	se3 = LSTM(256)(se2)
	# decoder model
	decoder1 = add([fe2, se3])
	decoder2 = Dense(256, activation='relu')(decoder1)
	outputs = Dense(vocab_size, activation='softmax')(decoder2)
	# tie it together [image, seq] [word]
	model = Model(inputs=[inputs1, inputs2], outputs=outputs)
	# compile model
	model.compile(loss='categorical_crossentropy', optimizer='adam')
	# summarize model
	model.summary()
	plot_model(model, to_file='model.png', show_shapes=True)
	return model

# load training dataset (6K)
filename = 'Flickr8k_text/Flickr_8k.trainImages.txt'
train = load_set(filename)
print('Dataset: %d' % len(train))
# descriptions
train_descriptions = load_clean_descriptions('descriptions.txt', train)
print('Descriptions: train=%d' % len(train_descriptions))
# photo features
train_features = load_photo_features('features.pkl', train)
print('Photos: train=%d' % len(train_features))
# prepare tokenizer
tokenizer = create_tokenizer(train_descriptions)
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary Size: %d' % vocab_size)
# determine the maximum sequence length
max_length = max_length(train_descriptions)
print('Description Length: %d' % max_length)
# prepare sequences
X1train, X2train, ytrain = create_sequences(tokenizer, max_length, train_descriptions, train_features, vocab_size)

# load test set
filename = 'Flickr8k_text/Flickr_8k.devImages.txt'
test = load_set(filename)
print('Dataset: %d' % len(test))
# descriptions
test_descriptions = load_clean_descriptions('descriptions.txt', test)
print('Descriptions: test=%d' % len(test_descriptions))
# photo features
test_features = load_photo_features('features.pkl', test)
print('Photos: test=%d' % len(test_features))
# prepare sequences
X1test, X2test, ytest = create_sequences(tokenizer, max_length, test_descriptions, test_features, vocab_size)

# define the model
model = define_model(vocab_size, max_length)
# define checkpoint callback
checkpoint = ModelCheckpoint('model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='min')
# fit model
model.fit([X1train, X2train], ytrain, epochs=20, verbose=2, callbacks=[checkpoint], validation_data=([X1test, X2test], ytest))

---

➡️ **Next / 下一步**: File 5 of 7